In [ ]:
import os
import time
from typing import List, Optional
from pinecone import Pinecone, ServerlessSpec
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import UnstructuredMarkdownLoader, TextLoader

from Langchain_learning_2 import documents

from PineCone_test.PineCone_test import records
from sentence_transformers.util import normalize_embeddings


class RAG_Service:
    def __init__(
            self,
            index_name: str,
            api_key: Optional[str] = None,
            model_name: str = "Qwen/Qwen3-VL-Embedding-2B",
            dimension: int = 1536,  # Qwen2-1.5B 默认维度
            metric: str = "cosine",
            region: str = "us-east-1"
    ):
        self.index_name = index_name
        self.api_key = api_key or os.getenv("PINECONE_API_KEY")
        self.dimension = dimension
        self.metric = metric
        self.region = region

        # 1. 初始化 Pinecone 客户端
        self.pc = Pinecone(api_key=self.api_key)

        # 2. 初始化本地 Qwen Embedding 模型
        print(f"正在加载嵌入模型: {model_name}...")
        self.embeddings = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'device': 'cuda'},
            encode_kwargs={'normalize_embeddings': True}
        )

        # 3. 初始化语义分块器
        self.text_splitter = SemanticChunker(self.embeddings, breakpoint_threshold_type="percentile")

        # 绑定索引操作对象
        if self.index_name in [idx.name for idx in self.pc.list_indexes()]:
            self.index = self.pc.Index(self.index_name)
        else:
            self.index = None

    def create_index(self):
        """创建索引，如果已存在则跳过"""
        if self.index_name not in [idx.name for idx in self.pc.list_indexes()]:
            print(f"正在创建索引: {self.index_name}...")
            self.pc.create_index(
                name=self.index_name,
                dimension=self.dimension,
                metric=self.metric,
                spec=ServerlessSpec(cloud="aws", region=self.region)
            )
            # 等待索引初始化完成
            while not self.pc.describe_index(self.index_name).status['ready']:
                time.sleep(1)

        self.index = self.pc.Index(self.index_name)
        print(f"索引 {self.index_name} 已就绪。")

    def add_Document(self, file_path: str, namespace: str = "default"):
        """载入 Markdown 文档，进行语义分块、向量化并上传"""
        if not self.index:
            raise ValueError("索引未初始化，请先调用 create_index()")

        print(f"正在处理文档: {file_path} ...")
        # 1. 加载 Markdown
        loader = UnstructuredMarkdownLoader(file_path)
        raw_docs = loader.load()

        # 2. 语义分块
        semantic_chunks = self.text_splitter.split_documents(raw_docs)

        # 3. 准备上传数据格式
        vectors_to_upsert = []
        for i, doc in enumerate(semantic_chunks):
            # 生成向量嵌入
            embedding = self.embeddings.embed_query(doc.page_content)

            # 这里的 ID 建议包含文件名以防重复
            doc_id = f"{os.path.basename(file_path)}_{i}"

            vectors_to_upsert.append({
                "id": doc_id,
                "values": embedding,
                "metadata": {
                    "text": doc.page_content,  # 必须存入原文以便搜索返回
                    "source": file_path
                }
            })

            # 分批上传，防止 payload 过大
            if len(vectors_to_upsert) >= 100:
                self.index.upsert(vectors=vectors_to_upsert, namespace=namespace)
                vectors_to_upsert = []

        if vectors_to_upsert:
            self.index.upsert(vectors=vectors_to_upsert, namespace=namespace)

        print(f"成功将 {len(semantic_chunks)} 个语义块上传至 Namespace: {namespace}")

    def Search_Vector(self, query: str, top_k: int = 3, namespace: str = "default"):
        """执行向量搜索并返回结果文本"""
        if not self.index:
            raise ValueError("索引未初始化")

        # 1. 向量化用户问题
        query_vector = self.embeddings.embed_query(query)

        # 2. 在 Pinecone 中检索
        search_results = self.index.query(
            vector=query_vector,
            top_k=top_k,
            namespace=namespace,
            include_metadata=True
        )

        # 3. 提取元数据中的文本内容
        context_list = []
        for res in search_results["matches"]:
            context_list.append({
                "text": res["metadata"]["text"],
                "score": res["score"],
                "source": res["metadata"]["source"]
            })

        return context_list

In [10]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
import re
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone_text.sparse import BM25Encoder

loader = TextLoader(
    "lawApp_LangGraph/MarkDownFiles/中国法院2019年度案例：婚姻家庭与继承纠纷.md",
    encoding="utf-8"
)

headers_to_split_on = [("#", "Header_1")]

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# 加载一次之后可以反复使用
# 稀疏向量
bm25 = BM25Encoder().load("lawApp_LangGraph/RAG_service/bm25_law_params.json")

# 密集向量
model_name = "BAAI/bge-large-zh-v1.5"
embeddings = HuggingFaceEmbeddings(model_name=model_name, encode_kwargs={'normalize_embeddings': True})

documents = loader.load()

for doc in documents:
    # split_text 返回 List[Document]，每个 Document 对应一个一级标题下的内容块
    articles = md_splitter.split_text(doc.page_content)

# 获取元数据和正文内容
for i, article in enumerate(articles):

    metadata = {}

    # 提取年份：假设文件名或路径包含 202X
    year_match = re.search(r"20\d{2}", article.page_content)
    if year_match:
        metadata["year"] = year_match.group(0) if year_match else "Unknown"

        # 提取裁判书字号：匹配如（2023）最高法民终...号
        case_num_pattern = r'裁判书字号[\s\\n]+((?:(?!裁判书字号)[\s\S])+?法院[\s\S]+?书)'
        case_num_match = re.search(case_num_pattern, article.page_content)

        metadata["case_number"] = case_num_match.group(1).strip() if case_num_match else "未识别"

        # 提取案由：通常在字号之后，或者是特定的段落
        case_cause_pattern = r"案由[:：]\s*([\u4e00-\u9fa5]+)"
        cause_match = re.search(case_cause_pattern, article.page_content)

        metadata["case_cause"] = cause_match.group(1) if cause_match else "通用"

        # 提取基本案情
        facts_pattern = r'【基本案情】\s*([\s\S]+?)(?=\n【|$)'
        facts_match = re.search(facts_pattern, article.page_content)

        # 获取捕获组内容（不含【基本案情】）
        raw_content = facts_match.group(1)
        # 去除空格、换行、制表符等所有空白字符，以及 # 符号
        facts_cleaned = re.sub(r'\n+', '\n', raw_content).strip()  # 去除所有空白（空格、换行等）
        facts_cleaned = facts_cleaned.replace('#', '')  # 去除所有 # 字符

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50,
    separators=["\n\n", "\n", "。", "；", "！", "？", " ", ""],  # 优先按段落切
    add_start_index=True
)

# 插入Pinecone数据容器
Pinecone_records = []

chunks = text_splitter.split_text(facts_cleaned)

for i, chunk in enumerate(chunks):
    record_id = f"doc_chunk_{i}"

    # 密集向量和稀疏向量
    dense_vector = embeddings.embed_query(chunk)

    # 返回 {"indices": [...], "values": [...]}
    sparse_vector = bm25.encode_documents(chunk)

    #
    """添加内容: id 向量数据 元数据:{年份 判决书 案由 文档切片}"""
    record = {
        "id": record_id,
        "values": dense_vector,  # 稠密向量列表 [0.12, -0.34, ...]
        "sparse_values": sparse_vector,  # 稀疏向量字典
        "metadata": {
            "chunk_index": i,
            **metadata,
            "chunk_text": chunk,
        }
    }

    Pinecone_records.append(record)





Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
print(len(Pinecone_records))

4


In [ ]:
from pinecone.grpc import PineconeGRPC as Pinecone

from dotenv import load_dotenv

load_dotenv()
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

# To get the unique host for an index,
# see https://docs.pinecone.io/guides/manage-data/target-an-index
index = pc.Index("pinecone-test-lawapp")

index.upsert(
    vectors=Pinecone_records,
    namespace="Law_test_namespace"
)

print(pc.describe_index("pinecone-test-lawapp"))

In [ ]:
{'_response_info': {'raw_headers': {'access-control-allow-origin': '*',
                                    'access-control-expose-headers': '*',
                                    'alt-svc': 'h3=":443"; ma=2592000',
                                    'content-length': '414',
                                    'content-type': 'application/json',
                                    'date': 'Sat, 25 Apr 2026 03:10:31 GMT',
                                    'server': 'Google Frontend',
                                    'vary': 'origin, '
                                            'access-control-request-method, '
                                            'access-control-request-headers',
                                    'via': '1.1 google',
                                    'x-cloud-trace-context': 'ef29f06e5f74fc059f8e58bade84201a',
                                    'x-pinecone-api-version': '2025-10'}},
 'deletion_protection': 'disabled',
 'dimension': 1024,
 'host': 'pinecone-test-lawapp-ldu4w3a.svc.aped-4627-b74a.pinecone.io',
 'metric': 'dotproduct',
 'name': 'pinecone-test-lawapp',
 'spec': {'serverless': {'cloud': 'aws',
                         'read_capacity': {'mode': 'OnDemand',
                                           'status': {'current_replicas': None,
                                                      'current_shards': None,
                                                      'state': 'Ready'}},
                         'region': 'us-east-1'}},
 'status': {'ready': True, 'state': 'Ready'},
 'tags': None,
 'vector_type': 'dense'}


In [ ]:
# 安装依赖：pip install langchain-huggingface sentence_transformers

from langchain_huggingface import HuggingFaceEmbeddings

model_name = "BAAI/bge-large-zh-v1.5"
embeddings = HuggingFaceEmbeddings(model_name=model_name)

# 单条文本嵌入
text = "这是一个测试文本。"
dense_vector = embeddings.embed_query(text)

# 多文本批量嵌入
texts = ["文本1", "文本2", "文本3"]
doc_vectors = embeddings.embed_documents(texts)
print(len(doc_vectors))

In [6]:
import re
import os
from pathlib import Path
from pinecone_text.sparse import BM25Encoder
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter


def extract_facts_from_md(content: str) -> list[str]:
    """
    从 Markdown 文本中提取所有一级标题下【基本案情】的内容
    返回清洗后的文本列表
    """
    headers_to_split_on = [("#", "Header_1")]
    md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
    articles = md_splitter.split_text(content)

    facts_list = []
    for article in articles:
        # 在每个案例中搜索【基本案情】
        facts_pattern = r'【基本案情】\s*([\s\S]+?)(?=\n【|$)'
        match = re.search(facts_pattern, article.page_content)
        if match:
            raw = match.group(1)
            cleaned = re.sub(r'\n+', ' ', raw).strip()  # 合并换行
            cleaned = cleaned.replace('#', '')
            facts_list.append(cleaned)

    return facts_list


# 1. 收集所有 .md 文件路径
md_dir = Path("lawApp_LangGraph/MarkDownFiles")
md_files = list(md_dir.glob("*.md"))
print(f"找到 {len(md_files)} 个Markdown文件")


找到 11 个Markdown文件


In [5]:

# 2. 提取所有“基本案情”文本
all_facts = []
for file_path in md_files:
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    facts = extract_facts_from_md(content)

    if facts:
        all_facts.extend(facts)
    else:
        print(f"警告：{file_path.name} 未找到【基本案情】段落")

print(f"成功提取 {len(all_facts)} 段基本案情")


成功提取 689 段基本案情


In [7]:

# 3. 一次性训练 BM25
bm25 = BM25Encoder()
bm25.fit(all_facts)  # 拟合IDF等参数

# 4. 保存训练好的参数（指定路径）
save_path = "lawApp_LangGraph/pdf_output/bm25_law_params.json"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
bm25.dump(save_path)

print(f"BM25 参数已保存至：{save_path}")

  0%|          | 0/689 [00:00<?, ?it/s]

BM25 参数已保存至：lawApp_LangGraph/pdf_output/bm25_law_params.json


In [14]:

pc_index = PC.Index("developer-quickstart-py")
pc_index.upsert_records(
    records=Pinecone_records,
    namespace="rag_test_namespace",
    field_mapping={
        "id": "id",
        "text": "chunk_text",  # 将你的 "content" 映射到内部的文本字段
        "metadata": [
            "chunk_index",
            "year",
            "case_number",
            "case_cause"
        ]
    }
)


TypeError: Index.upsert_records() got an unexpected keyword argument 'field_mapping'

In [6]:
from pinecone import Pinecone
import os
from dotenv import load_dotenv

load_dotenv()
Pinecone_api_key = os.getenv("PINECONE_API_KEY")
PC = Pinecone(api_key=Pinecone_api_key)
PC.describe_index("my-rag-index")

{
    "name": "my-rag-index",
    "metric": "cosine",
    "host": "my-rag-index-ldu4w3a.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "2

In [ ]:
# 上传文本数据和上传向量数据

from pinecone.grpc import PineconeGRPC as Pinecone

pc = Pinecone(api_key="YOUR_API_KEY")

# To get the unique host for an index,
# see https://docs.pinecone.io/guides/manage-data/target-an-index
index = pc.Index(host="INDEX_HOST")
"""

这里是上传密集向量

"""
index.upsert(
    vectors=[
        {
            "id": "A",
            "values": [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
            "metadata": {"genre": "comedy", "year": 2020}
        },
        {
            "id": "B",
            "values": [0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2],
            "metadata": {"genre": "documentary", "year": 2019}
        },
        {
            "id": "C",
            "values": [0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3],
            "metadata": {"genre": "comedy", "year": 2019}
        },
        {
            "id": "D",
            "values": [0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4],
            "metadata": {"genre": "drama"}
        }
    ],
    namespace="example-namespace"
)

In [ ]:
from pinecone import Pinecone, SparseValues, Vector

pc = Pinecone(api_key="YOUR_API_KEY")

# To get the unique host for an index,
# see https://docs.pinecone.io/guides/manage-data/target-an-index


"""

上传稀疏向量数据
相比密集向量
新增indices chunk_text
也就是说
上传稀疏向量同时需要添加文本和向量

"""
index = pc.Index(host="INDEX_HOST")

index.upsert(
    namespace="example-namespace",
    vectors=[
        {
            "id": "vec1",
            "sparse_values": {
                "values": [1.7958984, 0.41577148, 2.828125, 2.8027344, 2.8691406, 1.6533203, 5.3671875, 1.3046875,
                           0.49780273, 0.5722656, 2.71875, 3.0820312, 2.5019531, 4.4414062, 3.3554688],
                "indices": [822745112, 1009084850, 1221765879, 1408993854, 1504846510, 1596856843, 1640781426,
                            1656251611, 1807131503, 2543655733, 2902766088, 2909307736, 3246437992, 3517203014,
                            3590924191]
            },
            "metadata": {
                "chunk_text": "AAPL reported a year-over-year revenue increase, expecting stronger Q3 demand for its flagship phones.",
                "category": "technology",
                "quarter": "Q3"
            }
        },
        {
            "id": "vec2",
            "sparse_values": {
                "values": [0.4362793, 3.3457031, 2.7714844, 3.0273438, 3.3164062, 5.6015625, 2.4863281, 0.38134766,
                           1.25, 2.9609375, 0.34179688, 1.4306641, 0.34375, 3.3613281, 1.4404297, 2.2558594, 2.2597656,
                           4.8710938, 0.5605469],
                "indices": [131900689, 592326839, 710158994, 838729363, 1304885087, 1640781426, 1690623792, 1807131503,
                            2066971792, 2428553208, 2548600401, 2577534050, 3162218338, 3319279674, 3343062801,
                            3476647774, 3485013322, 3517203014, 4283091697]
            },
            "metadata": {
                "chunk_text": "Analysts suggest that AAPL'\''s upcoming Q4 product launch event might solidify its position in the premium smartphone market.",
                "category": "technology",
                "quarter": "Q4"
            }
        },
        {
            "id": "vec3",
            "sparse_values": {
                "values": [2.6875, 4.2929688, 3.609375, 3.0722656, 2.1152344, 5.78125, 3.7460938, 3.7363281, 1.2695312,
                           3.4824219, 0.7207031, 0.0826416, 4.671875, 3.7011719, 2.796875, 0.61621094],
                "indices": [8661920, 350356213, 391213188, 554637446, 1024951234, 1640781426, 1780689102, 1799010313,
                            2194093370, 2632344667, 2641553256, 2779594451, 3517203014, 3543799498, 3837503950,
                            4283091697]
            },
            "metadata": {
                "chunk_text": "AAPL'\''s strategic Q3 partnerships with semiconductor suppliers could mitigate component risks and stabilize iPhone production",
                "category": "technology",
                "quarter": "Q3"
            }
        },
        {
            "id": "vec4",
            "sparse_values": {
                "values": [0.73046875, 0.46972656, 2.84375, 5.2265625, 3.3242188, 1.9863281, 0.9511719, 0.5019531,
                           4.4257812, 3.4277344, 0.41308594, 4.3242188, 2.4179688, 3.1757812, 1.0224609, 2.0585938,
                           2.5859375],
                "indices": [131900689, 152217691, 441495248, 1640781426, 1851149807, 2263326288, 2502307765, 2641553256,
                            2684780967, 2966813704, 3162218338, 3283104238, 3488055477, 3530642888, 3888762515,
                            4152503047, 4177290673]
            },
            "metadata": {
                "chunk_text": "AAPL may consider healthcare integrations in Q4 to compete with tech rivals entering the consumer wellness space.",
                "category": "technology",
                "quarter": "Q4"
            }
        }
    ]
)

In [ ]:
import os
from pinecone import Pinecone, ServerlessSpec
from langchain_community.retrievers import (
    PineconeHybridSearchRetriever,
)
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone_text.sparse import BM25Encoder

# 1. 初始化环境变量
os.environ["PINECONE_API_KEY"] = "你的API_KEY"
index_name = "law-assistant-hybrid"

# 2. 初始化 Pinecone 客户端
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# 3. 创建索引 (如果不存在)
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1024,  # BGE-M3 维度
        metric="dotproduct",  # 混合检索必须用 dotproduct
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

# 4. 初始化稠密向量模型 (BGE-M3)
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# 5. 初始化稀疏向量编码器 (BM25)
# 在生产环境中，应先用你的法律语料库进行 .fit() 训练
bm25_encoder = BM25Encoder().default()
# 提示：实际项目中请运行 bm25_encoder.fit(all_legal_texts) 并保存 json

# 6. 构建 LangChain 检索器
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25_encoder,
    index=pc.Index(index_name),
    top_k=10,
    alpha=0.4  # 1偏向语义，0偏向关键词匹配
)

# 7. 执行查询
query = "寻衅滋事罪的量刑标准是什么？"
docs = retriever.invoke(query)

for doc in docs:
    print(f"找到条文：{doc.page_content[:100]}...")


## 一、婚约财产纠纷

# 1彩礼返还比例所参考的法律依据

##### ——马某甲诉李某甲等婚约财产案

##### 【案件基本信息】

1. 裁判书字号

河北省遵化市人民法院(2017)冀0281民初第5922号民事判决书

2. 案由: 婚约财产纠纷

3. 当事人

原告:马某甲

被告: 李某甲、李某乙、王某甲

【基本案情】

原告马某甲与被告李某甲于 2016年上半年通过网络结识后确立恋爱关系。2016年 8月 6日，马某甲、李某甲在双方父母主持下在原告家中举行定亲仪式并同居生活，当日原告给付被告彩礼款 10万元及黄金项链一条。双方定亲之前，被告李某甲于 2016年 7月经遵化市

第二医院确诊怀孕，并于 2016年 8月 1日、8月 2日、8月 15日、8月 16日、8月 23日在遵化市人民医院进行保胎治疗后流产。2017年 10月，原告马某甲与被告李某甲因定亲后结婚前的琐事产生矛盾，在双方家长交涉未果的情况下，原告马某甲于 2017年 11月 3日起诉。

##### 【案件焦点】

1. 彩礼的情况；2. 共同生活的情况；3. 双方未能进行婚姻登记的原因。

##### 【法院裁判要旨】

河北省遵化市人民法院经审理认为:彩礼是订立婚约的男女之间发生的财物往来。原告马某甲与被告李某甲举行定亲仪式后未办理结婚登记,原告马某甲请求返还按照习俗给付的彩礼,本院予以支持;婚约的当事人虽然是马某甲、李某甲,但婚约的订立及在订立过程中双方的父母均参与、主持,故原告马某甲要求被告李某甲与其父亲被告李某乙、母亲王某甲返还彩礼款符合法律规定,本院予以支持。原告主张给付被告彩礼款10万元的意见被告予以否认,认可给付彩礼款5万元,并主张其于2016年8月6日存入中国人民建设银行的存款10万元中有5万元是其母亲给付,但未提交证据佐证,其主张不足以对抗原告马某甲申请本院调取的中国人民建设银行的存款凭条记载内容的效力,本院予以认定原告马某甲给付的彩礼款为10万元。原告要求返还黄金项链、手镯共计价款13920元,被告李某甲对给付手镯的主张予以否认,对给付项链虽予以认可,但原告提交的销售商出具的收据,不是正式票据,且收据中未详细注明黄金项链及手镯的价格及质量,提交的微信聊天记录中关于“二金”的实物现状、给付、品质及存放地点的内容又不具有完整性、连续性,原告关于返还价值139

In [1]:
import re

text = """
1. 裁判书字号  \n北京市第三中级人民法院(2017)京03民终第8341号民事判决书
"""

pattern = r'\d+\.\s+裁判书字号\s*\n\s*(.+)'
matches = re.findall(pattern, text)

for m in matches:
    print(m)

北京市第三中级人民法院(2017)京03民终第8341号民事判决书


In [2]:
from transformers import EncoderDecoderCache, Trainer
from peft import PeftModel

print("导入成功！")

导入成功！


In [2]:
pip
install
weasyprint
markdown
pygments

Looking in indexes: http://mirrors.aliyun.com/pypi/simple/
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from datetime import datetime
import markdown
import os
from weasyprint import HTML


def markdown_to_pdf(markdown_text_filePath: str, filename: str = None) -> str:
    # 1. 读取 Markdown 文件
    with open(markdown_text_filePath, "r", encoding="utf-8") as f:
        markdown_text = f.read()

    # 2. Markdown → HTML（保留表格、代码高亮）
    html_body = markdown.markdown(
        markdown_text,
        extensions=['extra', 'codehilite']
    )

    # 3. 嵌入 CSS，确保中文排版
    styled_html = f"""
    <!DOCTYPE html>
    <html lang="zh-CN">
    <head>
        <meta charset="UTF-8">
        <style>
            @page {{
                size: A4;
                margin: 2cm;
            }}
            body {{
                font-family: 'SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei', sans-serif;
                font-size: 14px;
                line-height: 1.6;
            }}
            h1 {{ color: #333; }}
            pre {{ background: #f4f4f4; padding: 10px; border-radius: 5px; }}
            code {{ font-family: 'Courier New', monospace; }}
        </style>
    </head>
    <body>{html_body}</body>
    </html>
    """

    # 4. 生成文件名
    if not filename:
        filename = f"report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"

    # 5. 保存 PDF
    output_dir = "lawApp_LangGraph/pdf_output"
    os.makedirs(output_dir, exist_ok=True)
    file_path = os.path.join(output_dir, filename)

    HTML(string=styled_html).write_pdf(file_path)

    return f"PDF 已成功生成，文件路径：{file_path}"


def markdown_to_html(markdown_text: str) -> str:
    return markdown.markdown(markdown_text, extensions=['extra', 'codehilite'])


# 测试
print(markdown_to_pdf("lawApp_LangGraph/MarkDownFiles/中国法院2014年度案例_婚姻家庭与继承纠纷.md",
                      "中国法院2014年度案例_婚姻家庭与继承纠纷"))

In [4]:
import torch
import torchvision
import transformers

print(f"PyTorch版本: {torch.__version__}")
print(f"torchvision版本: {torchvision.__version__}")
print(f"transformers版本: {transformers.__version__}")
print(f"CUDA是否可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA版本: {torch.version.cuda}")

PyTorch版本: 2.6.0+cpu
torchvision版本: 0.21.0+cpu
transformers版本: 5.5.4
CUDA是否可用: False


In [ ]:
{'chunk_index': 0, 'id': 'doc_chunk_0', 'metadata': {'case_cause': '人身安全保护令',
                                                     'case_number': '云南省昭通市鲁甸县人民法院(2017)云0621民保令第2号民事裁定书',
                                                     'chunk_text': '申请人耿某某与被申请人曹某某均系教师，二人于2011年确立恋爱关系，于2013年10月14日办理结婚登记手续，于2014年5月18日生育一子曹某涵。婚后夫妻因家务琐事发生争吵，2015年9月、2016年5月，申请人耿某某先后两次被被申请人曹某某打伤。2017年9月23日，申请',
                                                     'year': '2017'},
 'sparse_values': {
     'indices': [1826951998, 3302586229, 4280047764, 2906185329, 3557921014, 1728204391, 550526999, 2068246531,
                 2306698243, 3251671578, 287991880, 1449754254, 2694352034],
     'values': [0.6625503854497659, 0.6625503854497659, 0.6625503854497659, 0.6625503854497659, 0.6625503854497659,
                0.6625503854497659, 0.6625503854497659, 0.6625503854497659, 0.6625503854497659, 0.6625503854497659,
                0.6625503854497659, 0.6625503854497659, 0.6625503854497659]
 },
 'values': [-0.03657614812254906, 0.04955880716443062, 0.04295945540070534, 0.016933824867010117, 0.019821351394057274,
            0.0 ** 14890265465, 0.016650019213557243,
            0.015882203355431557, -0.009750094264745712, 0.005802732892334461, 0.002266672905534506,
            -0.011243334040045738, -0.011120816692709923, -0.016657091677188873, 0.001065897406078875,
            0.012090726755559444, -0.02417769469320774, 0.008060940541327, 0.03416145592927933, -0.004575522616505623,
            -0.02823040261864662, -0.02307465858757496]}